In [ ]:
!pip install gensim

In [ ]:
import numpy as np
def cosine_similarity(a, b):
  return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
good = np.array([0.82, 0.41, 0.10])
great = np.array([0.79, 0.38, 0.12])
bad = np.array([-0.71, 0.12, -0.05])
print(f'good vs great : {cosine_similarity(good, great):.3f}')
print(f'good vs bad : {cosine_similarity(good, bad):.3f}')
print(f'great vs bad : {cosine_similarity(great, bad):.3f}')

In [ ]:
from gensim.models import Word2Vec
sentences = [
'the king ruled the kingdom wisely',
'the queen ruled the land gracefully',
'the prince will become king one day',
'a man walked into the palace',
'a woman walked into the palace',
'the dog barked loudly in the garden',
'the puppy played happily in the garden',
]
tokenised = [s.split() for s in sentences]
model = Word2Vec(sentences=tokenised, vector_size=50,
window=3, min_count=1, sg=1, epochs=200)
print(model.wv.most_similar('king', topn=3))
result = model.wv.most_similar(
positive=['dog', 'puppy'],
negative=['man'],
topn=2
)
print('king - man + woman =', result[1][0])

In [ ]:
!rm -f glove.6B.zip
!curl -O https://nlp.stanford.edu/data/glove.6B.zip
!unzip -o -q glove.6B.zip
from gensim.scripts.glove2word2vec import glove2word2vec
glove2word2vec('glove.6B.100d.txt', 'glove.6B.100d.w2v.txt')
from gensim.models import KeyedVectors
glove = KeyedVectors.load_word2vec_format(
'glove.6B.100d.w2v.txt', binary=False
)
print(glove.most_similar('bollywood', topn=5))
res = glove.most_similar(positive=['paris','tamilnadu'], negative=['france'],
topn=1)
print('paris - france + india =', res[0][0])
print(glove.similarity('dog', 'puppy'))
print(glove.similarity('cat', 'cricket'))

In [ ]:
import gensim.downloader as api
ft = api.load('fasttext-wiki-news-subwords-300')
print(ft.most_similar('bollywoodesque', topn=3))

[==================================================] 100.0% 958.5/958.4MB downloaded


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from gensim.models import KeyedVectors



words = [
    'king', 'queen', 'man', 'woman', 'prince', 'princess',
    'dog', 'puppy', 'cat', 'kitten','rama chandra','shravanth'
    'paris', 'france', 'delhi', 'india', 'london', 'england'
]

valid_words = [w for w in words if w in glove.key_to_index]

missing_words = set(words) - set(valid_words)
if missing_words:
    print("Missing words:", missing_words)

vectors = np.array([glove[w] for w in valid_words])

print("Vector matrix shape:", vectors.shape)

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(
    coords[:, 0],
    coords[:, 1],
    s=80,
    color='steelblue',
    zorder=3
)

for i, word in enumerate(missing_words):
    ax.annotate(
        word,
        (coords[i, 0], coords[i, 1]),
        fontsize=11,
        xytext=(5, 4),
        textcoords='offset points'
    )

ax.set_title(
    'GloVe Word Clusters (PCA Projection to 2D)',
    fontsize=13
)

ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.savefig('word_clusters.png', dpi=150)
plt.show()

In [ ]:
import numpy as np

def sentence_vector(sentence, model, dim=100):

    tokens = sentence.lower().split()

    vectors = []

    for token in tokens:
        if token in model:
            vectors.append(model[token])

    if not vectors:
        return np.zeros(dim)

    return np.mean(vectors, axis=0)


s1 = 'man bite the dog at morning'
s2 = 'dog bite the man at afternoon'
s3 = 'A romantic comedy about falling in love in Paris'

v1 = sentence_vector(s1, glove)
v2 = sentence_vector(s2, glove)
v3 = sentence_vector(s3, glove)

from numpy.linalg import norm

cos = lambda a, b: np.dot(a, b) / (norm(a) * norm(b))

print(f's1 vs s2 (both action): {cos(v1, v2):.3f}')
print(f's1 vs s3 (action vs romance): {cos(v1, v3):.3f}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

df = pd.DataFrame({
    "description": [
        "A brave warrior saves humanity from destruction",
        "A hero fights evil forces to protect the world",
        "Two strangers fall in love in Paris",
        "A romantic story about relationships and emotions",
        "A hilarious comedy full of funny situations",
        "Friends experience a series of humorous adventures",
        "A detective investigates a mysterious murder case",
        "A criminal conspiracy shocks the entire city"
    ],
    "genre": [
        "action",
        "action",
        "romance",
        "romance",
        "comedy",
        "comedy",
        "crime",
        "crime"
    ]
})
X = np.array([sentence_vector(desc, glove) for desc in df['description']])
y = df['genre']
X_tr, X_te, y_tr, y_te = train_test_split(
X, y, test_size=0.5, random_state=42, stratify=y
)
clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X_tr, y_tr)
print(classification_report(y_te, clf.predict(X_te)))

In [ ]:
import numpy as np
import pandas as pd
movies = [
{'title':'Dilwale Dulhania Le Jayenge',
'desc':'A young man follows the woman he loves to India after they fall inlove in Europe'},
{'title':'Lagaan',
'desc':'Villagers challenge British colonisers to a cricket match to avoipaying heavy taxes'},
{'title':'Dangal',
'desc':'A father trains his daughters to become world class wrestlers against all odds'},
{'title':'3 Idiots',
'desc':'Three college friends question the pressure of engineering education in India'},
{'title':'Dil Chahta Hai',
'desc':'Three best friends navigate love relationships and life aftergraduation'},
{'title':'Titanic',
'desc':'A wealthy girl and a poor artist fall in love on a doomed oceanliner'},
{'title':'Rocky',
'desc':'An underdog boxer trains relentlessly for a shot at the worldeavyweight champion'},
{'title':'Good Will Hunting',
'desc':'A genius janitor from Boston works through emotional trauma with aherapist'},
{'title':'Chariots of Fire',
'desc':'Two British athletes train and compete for glory in the 1924lympic Games'},
{'title':'Before Sunrise',
'desc':'Two strangers meet on a train in Europe and spend one romanticight in Vienna'},
]
df = pd.DataFrame(movies)
print(df[['title']].to_string())

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def sentence_vector(text, model, dim=100):
  tokens = text.lower().split()
  vecs = [model[t] for t in tokens if t in model]
  return np.mean(vecs, axis=0) if vecs else np.zeros(dim)
df['vector'] = df['desc'].apply(lambda d: sentence_vector(d, glove))
V = np.stack(df['vector'].values)
def find_similar(query_desc, df, top_n=3):
  qvec = sentence_vector(query_desc, glove).reshape(1, -1)
  sims = cosine_similarity(qvec, V)[0]
  idx = np.argsort(sims)[::-1][:top_n]
  result = df.iloc[idx][['title']].copy()
  result['similarity'] = np.round(sims[idx], 3)
  return result
q1 = 'An underdog athlete trains hard and wins against all expectations'
print('QUERY:', q1)
print(find_similar(q1, df).to_string(index=False))
q2 = 'Two people fall in love while travelling across Europe'
print('\nQUERY:', q2)
print(find_similar(q2, df).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(V)
colours = ['#7F77DD']*5 + ['#1D9E75']*5
fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(coords[:,0], coords[:,1], c=colours, s=120, zorder=3)
for i, row in df.iterrows():
ax.annotate(row['title'], coords[i], fontsize=8.5,
xytext=(6, 4), textcoords='offset points')
import matplotlib.patches as mpatches
ax.legend(handles=[
mpatches.Patch(color='#7F77DD', label='Bollywood'),
mpatches.Patch(color='#1D9E75', label='Hollywood'),
], fontsize=10)
ax.set_title('Movie description clusters (GloVe + PCA)', fontsize=13)
plt.tight_layout()
plt.savefig('movie_clusters.png', dpi=150)
plt.show()